# THRawS — Result evaluation

- **Metrics table** with threshold of 0.5: accuracy, precision, recall, F1
- **ROC curve + AUROC**: over threshold
- **F1 vs threshold curve**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, classification_report
)

bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/THRawS'

In [ ]:
y_test        = np.load(os.path.join(bewerkte_data_dir, 'y_test_THRawS.npy'))
y_prob_majority = np.load(os.path.join(bewerkte_data_dir, 'y_prob_majority_THRawS.npy'))
y_prob_rf     = np.load(os.path.join(bewerkte_data_dir, 'y_prob_rf_THRawS.npy'))
y_prob_xgb    = np.load(os.path.join(bewerkte_data_dir, 'y_prob_xgb_THRawS.npy'))

print(f"Test samples: {len(y_test)}")
print(f"Klasse 0 (niet-burnt): {(y_test==0).sum()}, Klasse 1 (burnt): {(y_test==1).sum()}")

# Metrics table <br>
with threshold of 0.5 (standard threshold)

In [ ]:
def metrics_with_threshold(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1 (binary)': f1_score(y_true, y_pred, zero_division=0),
        'F1 (weighted)': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'AUROC':     roc_auc_score(y_true, y_prob)
    }

models = {
    'Majority Class': y_prob_majority,
    'Random Forest':  y_prob_rf,
    'XGBoost':        y_prob_xgb,
}

print("\n=== THRawS Model Performance at Threshold 0.5 ===")
print(f"{'Model':<20} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1 bin':>8} {'F1 wtd':>8} {'AUROC':>7}")
print("-" * 76)

for name, probs in models.items():
    m = metrics_with_threshold(y_test, probs, threshold=0.5)
    print(f"{name:<20} {m['Accuracy']:>9.3f} {m['Precision']:>10.3f} {m['Recall']:>8.3f} "
          f"{m['F1 (binary)']:>8.3f} {m['F1 (weighted)']:>8.3f} {m['AUROC']:>7.3f}")


## ROC Curve + AUROC

Shows trade-off between TPR and FPR over threshold. The AUROC shows this in 1 number:
- AUROC = 1.0 → perfect model
- AUROC = 0.5 → random guessing

**Note for majority class:** because all samples have the same chance there is only one point on the ROC curve: 0.5.

In [ ]:
colors = {
    'Majority Class': ('tab:green', '--'),
    'Random Forest':  ('royalblue', '-'),
    'XGBoost':        ('darkorange', '-'),
}

plt.figure(figsize=(12, 6))

for name, probs in models.items():
    color, style = colors[name]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auroc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, color=color, linestyle=style,
             label=f'{name} (AUROC = {auroc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], color='gray', linestyle=':', linewidth=1.5, label='Random (AUROC = 0.5)')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('THRawS Results: ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## F1 Score vs Threshold

**Majority class:** because all samples get the same chance, curve has one value:
- Threshold < prior → everything predicted as 1 → high recall, low precision
- Threshold > prior → everything predicted as 0 → F1 = 0

In [ ]:
thresholds = np.arange(0.05, 1.0, 0.05)

def f1_per_threshold_binary(y_true, y_prob, thresholds):
    return [
        f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]

plt.figure(figsize=(12, 7))

for name, probs in models.items():
    color, style = colors[name]
    scores = f1_per_threshold_binary(y_test, probs, thresholds)
    max_f1 = max(scores)
    plt.plot(thresholds, scores, color=color, linestyle=style,
             label=f'{name} (Max F1: {max_f1:.2f})', marker='o', linewidth=2)

plt.axvline(0.5, color='gray', linestyle='--', label='Standaard Threshold (0.5)')
plt.title('THRawS Results: F1 Score vs Threshold', fontsize=14, fontweight='bold')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('F1 Score (Binary)', fontsize=12)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

## Precision & Recall vs Threshold

Shows trade-off between precision and recall with a changing threshold.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(16, 5), sharey=True)

for ax, (name, probs) in zip(axes, models.items()):
    color = colors[name][0]
    precisions = [precision_score(y_test, (probs >= t).astype(int), zero_division=0) for t in thresholds]
    recalls    = [recall_score(y_test,    (probs >= t).astype(int), zero_division=0) for t in thresholds]

    ax.plot(thresholds, precisions, color=color,    label='Precision', linewidth=2)
    ax.plot(thresholds, recalls,    color=color, linestyle='--', label='Recall', linewidth=2)
    ax.axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

axes[0].set_ylabel('Score', fontsize=12)
fig.suptitle('THRawS: Precision & Recall vs Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(f"y_test klasse verdeling: {np.unique(y_test, return_counts=True)}")

from sklearn.metrics import roc_auc_score
print(f"AUROC RF (uit bestand): {roc_auc_score(y_test, y_prob_rf):.3f}")

# Quick sanity: train een fris model en vergelijk
from sklearn.ensemble import RandomForestClassifier
X_train = np.load(os.path.join(bewerkte_data_dir, 'X_train_THRawS.npy'))
y_train = np.load(os.path.join(bewerkte_data_dir, 'y_train_THRawS.npy'))
X_test  = np.load(os.path.join(bewerkte_data_dir, 'X_test_THRawS.npy'))

print(f"y_train klasse verdeling: {np.unique(y_train, return_counts=True)}")
print(f"Komt y_test uit dit bestand overeen met de geladen y_test? {np.array_equal(y_test, np.load(os.path.join(bewerkte_data_dir, 'y_test_THRawS.npy')))}")

## Final classification report with threshold of 0.5 (standard)

In [ ]:
print("\n=== THRawS Model Performance at Threshold 0.5 ===")
for name, probs in models.items():
    y_pred = (probs >= 0.5).astype(int)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['NE (0)', 'E (1)'], zero_division=0))
    print()

## Final classification report of 0.85

In [ ]:
print("\n=== THRawS Model Performance at Threshold 0.85 ===")
for name, probs in models.items():
    y_pred = (probs >= 0.85).astype(int)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['NE (0)', 'E (1)'], zero_division=0))
    print()

## Threshold sweep vs. resampling per threshold

Twee verschillende manieren om het effect van threshold te onderzoeken:

- **Threshold sweep** (grafieken hierboven): één model trainen met gebalanceerde
  class weights, en de afkapwaarde achteraf verschuiven bij het voorspellen.
- **Resampling per threshold**: voor elke threshold een nieuw model trainen op
  een dataset waarin de klasse-verhouding is aangepast (over-/undersampling van
  de brand-klasse), en steeds evalueren op de neutrale threshold 0.5.

Bij resampling ziet het model voor elke threshold daadwerkelijk andere data,
in tegenstelling tot `class_weight`, die alleen de splitsingscriteria binnen de
boom aanpast zonder de data zelf te veranderen.

In [ ]:
rf_results_resample  = np.load(os.path.join(bewerkte_data_dir, 'results_rf_resample_THRawS.npy'),  allow_pickle=True)
xgb_results_resample = np.load(os.path.join(bewerkte_data_dir, 'results_xgb_resample_THRawS.npy'), allow_pickle=True)

print(f"Geladen: {len(rf_results_resample)} resampling-resultaten voor RF en XGBoost")

In [ ]:
thresholds_resample = [r['threshold'] for r in rf_results_resample]

def f1_per_threshold(y_true, y_prob, thresholds):
    """Berekent binaire F1 score voor elke threshold (zelfstandige versie,
    onafhankelijk van eerdere cellen in deze notebook)."""
    return [
        f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]

thresholds = np.arange(0.05, 1.0, 0.05)

# F1 uit de threshold sweep (dezelfde probs, threshold toegepast bij voorspellen)
rf_f1_sweep  = f1_per_threshold(y_test, y_prob_rf,  thresholds)
xgb_f1_sweep = f1_per_threshold(y_test, y_prob_xgb, thresholds)

# F1 uit resampling (apart getraind model per threshold, geëvalueerd op 0.5)
rf_f1_resample  = [r['f1'] for r in rf_results_resample]
xgb_f1_resample = [r['f1'] for r in xgb_results_resample]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

axes[0].plot(thresholds, rf_f1_sweep, marker='o', color='royalblue',
             label='Threshold sweep', linewidth=2)
axes[0].plot(thresholds_resample, rf_f1_resample, marker='s', color='royalblue',
             linestyle='--', label='Resampling', linewidth=2, alpha=0.7)
axes[0].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[0].set_title('Random Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('F1 Score')
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

axes[1].plot(thresholds, xgb_f1_sweep, marker='o', color='darkorange',
             label='Threshold sweep', linewidth=2)
axes[1].plot(thresholds_resample, xgb_f1_resample, marker='s', color='darkorange',
             linestyle='--', label='Resampling', linewidth=2, alpha=0.7)
axes[1].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[1].set_title('XGBoost', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

fig.suptitle('THRawS: Threshold Sweep vs. Resampling per Threshold (F1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Threshold':>10} {'RF sweep':>10} {'RF resample':>12} {'XGB sweep':>10} {'XGB resample':>13}")
print("-" * 60)
for i, t in enumerate(thresholds_resample):
    print(f"  {t:>8.2f} {rf_f1_sweep[i]:>10.3f} {rf_f1_resample[i]:>12.3f} "
          f"{xgb_f1_sweep[i]:>10.3f} {xgb_f1_resample[i]:>13.3f}")